<a href="https://colab.research.google.com/github/F1ameX/Modern-Methods-of-Deep-Machine-Learning/blob/main/4_CNN/4_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install -q optuna

In [29]:
import os
import math
import random
from collections import Counter
from dataclasses import dataclass
from typing import Dict, Any, Tuple, List, Optional

import optuna
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms

In [30]:
class ConvBlockA(nn.Module):
    def __init__(
        self,
        input_channels: int,
        output_channels: int,
        conv_kernel_size: int,
        conv_stride: int,
        pool_kernel_size: int,
        pool_stride: int,
    ):
        super().__init__()

        self.conv = nn.Conv2d(
            in_channels=input_channels,
            out_channels=output_channels,
            kernel_size=conv_kernel_size,
            stride=conv_stride,
            padding=conv_kernel_size // 2,
        )
        self.activation = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=pool_kernel_size, stride=pool_stride)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        x = self.activation(x)
        x = self.pool(x)
        return x


class ConvBlockB(nn.Module):
    def __init__(
        self,
        input_channels: int,
        output_channels: int,
        conv_kernel_size: int,
        conv_stride: int,
        pool_kernel_size: int,
        pool_stride: int,
    ):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=input_channels,
            out_channels=output_channels,
            kernel_size=conv_kernel_size,
            stride=conv_stride,
            padding=conv_kernel_size // 2,
        )
        self.conv2 = nn.Conv2d(
            in_channels=output_channels,
            out_channels=output_channels,
            kernel_size=conv_kernel_size,
            stride=1,
            padding=conv_kernel_size // 2,
        )

        self.activation = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=pool_kernel_size, stride=pool_stride)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.activation(x)

        x = self.conv2(x)
        x = self.activation(x)

        x = self.pool(x)
        return x


class CNN(nn.Module):
    def __init__(
        self,
        block_type: str = "a",
        n_blocks: int = 2,
        base_channels: int = 16,
        conv_kernel_size: int = 3,
        conv_stride: int = 1,
        pool_kernel_size: int = 2,
        pool_stride: int = 2,
        dropout: float = 0.0,
        num_classes: int = 10,
        kernel_size: int | None = None,
        pool_kernel: int | None = None,
        base_channels_alias: int | None = None,
        n_blocks_alias: int | None = None,
        pool_stride_alias: int | None = None,
        conv_stride_alias: int | None = None,
    ):
        super().__init__()
        assert block_type in ("a", "b")
        assert n_blocks >= 1

        if kernel_size is not None:
            conv_kernel_size = kernel_size
        if pool_kernel is not None:
            pool_kernel_size = pool_kernel
        if base_channels_alias is not None:
            base_channels = base_channels_alias
        if n_blocks_alias is not None:
            n_blocks = n_blocks_alias
        if pool_stride_alias is not None:
            pool_stride = pool_stride_alias
        if conv_stride_alias is not None:
            conv_stride = conv_stride_alias

        BlockClass = ConvBlockA if block_type == "a" else ConvBlockB

        blocks = []
        input_channels = 1
        channels = base_channels

        for _ in range(n_blocks):
            blocks.append(
                BlockClass(
                    input_channels=input_channels,
                    output_channels=channels,
                    conv_kernel_size=conv_kernel_size,
                    conv_stride=conv_stride,
                    pool_kernel_size=pool_kernel_size,
                    pool_stride=pool_stride,
                )
            )
            input_channels = channels
            channels *= 2

        self.features = nn.Sequential(*blocks)

        with torch.no_grad():
            dummy_x = torch.zeros(1, 1, 28, 28)
            try:
                dummy_out = self.features(dummy_x)
            except RuntimeError as e:

                raise ValueError("Invalid geometry: conv/pool params collapse 28x28 to 0x0") from e
            flatten_dim = dummy_out.view(1, -1).shape[1]

        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.classifier = nn.Linear(flatten_dim, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = x.flatten(1)
        x = self.dropout(x)
        logits = self.classifier(x)
        return logits

In [31]:
SEED = 52
BATCH_SIZE = 128

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

full_train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_size = 50_000
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

len(train_dataset), len(val_dataset), len(test_dataset)

(50000, 10000, 10000)

In [32]:
rng = np.random.default_rng(SEED)


def stratified_indices_from_targets(targets, n_per_class: int, seed: int = 52):
    rng_local = np.random.default_rng(seed)
    targets = np.asarray(targets, dtype=int)
    idx_by_class = [np.where(targets == c)[0] for c in range(10)]
    chosen = []
    for c in range(10):
        if len(idx_by_class[c]) < n_per_class:
            raise ValueError(f"Not enough samples for class {c}: have {len(idx_by_class[c])}, need {n_per_class}")
        chosen_c = rng_local.choice(idx_by_class[c], size=n_per_class, replace=False)
        chosen.append(chosen_c)
    chosen = np.concatenate(chosen)
    rng_local.shuffle(chosen)
    return chosen.tolist()

def get_targets(ds):
    if hasattr(ds, "targets"):
        t = ds.targets
        return t.cpu().numpy() if torch.is_tensor(t) else np.asarray(t)
    return np.array([ds[i][1] for i in range(len(ds))], dtype=int)

mini_train_per_class = 300
mini_val_per_class   = 100

train_targets = get_targets(train_dataset)
test_targets  = get_targets(test_dataset)

mini_train_idx = stratified_indices_from_targets(train_targets, mini_train_per_class, seed=SEED)
mini_val_idx   = stratified_indices_from_targets(test_targets,  mini_val_per_class,   seed=SEED)

mini_train_ds = Subset(train_dataset, mini_train_idx)
mini_val_ds   = Subset(test_dataset,  mini_val_idx)

def class_counts_from_subset(subset, targets_full):
    idx = np.array(subset.indices, dtype=int)
    return Counter(targets_full[idx].tolist())

print("Mini-train size:", len(mini_train_ds))
print("Mini-val size:  ", len(mini_val_ds))

print("Mini-train class counts:", dict(sorted(class_counts_from_subset(mini_train_ds, train_targets).items())))
print("Mini-val class counts:  ", dict(sorted(class_counts_from_subset(mini_val_ds, test_targets).items())))

Mini-train size: 3000
Mini-val size:   1000
Mini-train class counts: {0: 300, 1: 300, 2: 300, 3: 300, 4: 300, 5: 300, 6: 300, 7: 300, 8: 300, 9: 300}
Mini-val class counts:   {0: 100, 1: 100, 2: 100, 3: 100, 4: 100, 5: 100, 6: 100, 7: 100, 8: 100, 9: 100}


In [33]:
batch_size_mini = 256

mini_train_loader = DataLoader(
    mini_train_ds,
    batch_size=batch_size_mini,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

mini_val_loader = DataLoader(
    mini_val_ds,
    batch_size=batch_size_mini,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

x0, y0 = next(iter(mini_train_loader))
print("Mini-train batch:", x0.shape, y0.shape, "labels sample:", y0[:10].tolist())

Mini-train batch: torch.Size([256, 1, 28, 28]) torch.Size([256]) labels sample: [9, 6, 8, 1, 4, 8, 2, 5, 8, 5]


In [34]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    total = 0

    for images, labels in loader:
        optimizer.zero_grad()
        logits = model(images)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()

        bs = labels.size(0)
        total_loss += float(loss.item()) * bs
        total += bs

    return total_loss / total


@torch.no_grad()
def evaluate_acc(model, loader):
    model.eval()
    correct = 0
    total = 0
    for images, labels in loader:
        logits = model(images)
        preds = logits.argmax(dim=1)
        correct += int((preds == labels).sum().item())
        total += int(labels.numel())
    return correct / total


def quick_train_eval(
    model,
    train_loader,
    val_loader,
    *,
    learning_rate: float = 1e-3,
    weight_decay: float = 0.0,
    epochs: int = 3,
):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    history = []
    for ep in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer)
        val_acc = evaluate_acc(model, val_loader)
        history.append((ep, train_loss, val_acc))
    return history, history[-1][2]


model = CNN(
    block_type="a",
    n_blocks=2,
    base_channels=16,
    kernel_size=3,
    conv_stride=1,
    pool_kernel=2,
    pool_stride=2,
    dropout=0.1,
)

hist, acc = quick_train_eval(
    model,
    mini_train_loader,
    mini_val_loader,
    learning_rate=1e-3,
    weight_decay=0.0,
    epochs=3,
)

print("history (epoch, train_loss, val_acc):")
for row in hist:
    print(row)
print("final val_acc:", acc)

history (epoch, train_loss, val_acc):
(1, 2.000084377924601, 0.794)
(2, 1.0143744260470073, 0.847)
(3, 0.5106013103326161, 0.892)
final val_acc: 0.892


In [36]:
# ===== Cell 7 (FIXED): Optuna objective + study =====

def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 5e-3, log=True)
    weight_decay  = trial.suggest_float("weight_decay", 0.0, 1e-3)

    block_type    = trial.suggest_categorical("block_type", ["a", "b"])
    n_blocks      = trial.suggest_int("n_blocks", 1, 4)
    kernel_size   = trial.suggest_categorical("kernel_size", [3, 5])
    conv_stride   = trial.suggest_categorical("conv_stride", [1, 2])
    pool_kernel   = trial.suggest_categorical("pool_kernel", [2, 3])
    pool_stride   = trial.suggest_categorical("pool_stride", [2, 3])
    base_channels = trial.suggest_categorical("base_channels", [8, 16, 32])
    dropout       = trial.suggest_float("dropout", 0.0, 0.4)

    # ВАЖНО: падение происходит внутри CNN.__init__ на dummy-прогоне -> ловим тут
    try:
        model = CNN(
            block_type=block_type,
            n_blocks=n_blocks,
            base_channels=base_channels,
            kernel_size=kernel_size,
            conv_stride=conv_stride,
            pool_kernel=pool_kernel,
            pool_stride=pool_stride,
            dropout=dropout,
        )
    except (RuntimeError, ValueError):
        raise optuna.TrialPruned()

    # Быстрое обучение на мини-выборке
    _, val_acc = quick_train_eval(
        model,
        mini_train_loader,
        mini_val_loader,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        epochs=3,
    )

    trial.report(val_acc, step=0)
    if trial.should_prune():
        raise optuna.TrialPruned()

    return val_acc


study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
)

# чтобы даже если где-то всё равно вылезет runtime, Optuna не уронила прогон целиком
study.optimize(objective, n_trials=25, catch=(RuntimeError, ValueError))

print("best value (val_acc):", study.best_value)
print("best params:", study.best_params)

[I 2026-03-30 04:07:07,060] A new study created in memory with name: no-name-08c6eb83-1a55-4986-aced-88c14757d284
[I 2026-03-30 04:07:07,083] Trial 0 pruned. 
[I 2026-03-30 04:07:18,831] Trial 1 finished with value: 0.844 and parameters: {'learning_rate': 0.0002840027973938916, 'weight_decay': 0.00042855107052529827, 'block_type': 'a', 'n_blocks': 2, 'kernel_size': 3, 'conv_stride': 1, 'pool_kernel': 2, 'pool_stride': 2, 'base_channels': 32, 'dropout': 0.04829799235629917}. Best is trial 1 with value: 0.844.
[I 2026-03-30 04:07:23,494] Trial 2 finished with value: 0.867 and parameters: {'learning_rate': 0.0019561974909572065, 'weight_decay': 0.00016353642601897467, 'block_type': 'a', 'n_blocks': 3, 'kernel_size': 3, 'conv_stride': 1, 'pool_kernel': 3, 'pool_stride': 3, 'base_channels': 16, 'dropout': 0.13168338236800864}. Best is trial 2 with value: 0.867.
[I 2026-03-30 04:07:28,262] Trial 3 finished with value: 0.369 and parameters: {'learning_rate': 0.00024625175009067735, 'weight_de

best value (val_acc): 0.962
best params: {'learning_rate': 0.0019379759711812983, 'weight_decay': 0.00023075709565371485, 'block_type': 'b', 'n_blocks': 1, 'kernel_size': 3, 'conv_stride': 1, 'pool_kernel': 3, 'pool_stride': 2, 'base_channels': 32, 'dropout': 0.27041848818155734}


In [ ]:
trials_df = study.trials_dataframe(attrs=("number", "value", "state", "params"))
trials_df = trials_df.sort_values("value", ascending=False).reset_index(drop=True)

display(trials_df.head(10))

best_params = study.best_params
best_val_acc = study.best_value

print("best_val_acc:", best_val_acc)
print("best_params:", best_params)

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def build_best_model(params: dict):
    kw = dict(
        block_type=params["block_type"],
        n_blocks=params["n_blocks"],
        base_channels=params["base_channels"],
        kernel_size=params["kernel_size"],
        conv_stride=params["conv_stride"],
        pool_kernel=params["pool_kernel"],
        pool_stride=params["pool_stride"],
        num_classes=10,
    )
    if "dropout" in params:
        try:
            return CNN(**kw, dropout=params["dropout"])
        except TypeError:
            return CNN(**kw)
    return CNN(**kw)

def train_one_epoch_device(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    total = 0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()

        bs = labels.size(0)
        total_loss += float(loss.item()) * bs
        total += bs

    return total_loss / total

@torch.no_grad()
def evaluate_device(model, loader):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model(images)
        loss = F.cross_entropy(logits, labels, reduction="sum")
        total_loss += float(loss.item())

        preds = logits.argmax(dim=1)
        correct += int((preds == labels).sum().item())
        total += int(labels.numel())

    return {"loss": total_loss / total, "acc": correct / total}

def fit_full(
    model,
    train_loader,
    val_loader,
    *,
    learning_rate: float,
    weight_decay: float = 0.0,
    max_epochs: int = 20,
    target_acc: float = 0.90,
):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    first_epoch_reach = None
    history = []

    for ep in range(1, max_epochs + 1):
        train_loss = train_one_epoch_device(model, train_loader, optimizer)
        val_metrics = evaluate_device(model, val_loader)

        history.append((ep, train_loss, val_metrics["loss"], val_metrics["acc"]))

        if first_epoch_reach is None and val_metrics["acc"] >= target_acc:
            first_epoch_reach = ep

        print(f"epoch={ep:02d} | train_loss={train_loss:.4f} | val_loss={val_metrics['loss']:.4f} | val_acc={val_metrics['acc']:.4f}")

    return model, history, first_epoch_reach

final_model = build_best_model(best_params)
final_model, history, first_epoch_reach = fit_full(
    final_model,
    train_loader,
    val_loader,
    learning_rate=best_params["learning_rate"],
    weight_decay=best_params.get("weight_decay", 0.0),
    max_epochs=20,
    target_acc=0.90,
)

test_metrics = evaluate_device(final_model, test_loader)

print("\nFirst epoch reaching val_acc >= 0.90:", first_epoch_reach)
print("Final test_acc:", test_metrics["acc"])
print("Final test_loss:", test_metrics["loss"])